In [7]:
import torch
import random
import matplotlib.pyplot as plt
import numpy as np
from env import (SingleLeafThreadEnv, ThreadingConfig)
from loss import TrajectoryBalanceLoss
from viz import draw_tree_edge_index, draw_local_tree_sequence
from utils import build_backbone_segments_from_reference, get_max_actions, load_tskit_threading_inputs
from models import Policy

In [8]:
# TREES_PATH = "path/to/file.trees"
# TIME_GRID_SIZE = 20

# GENO, REFERENCE_FULL_TREES, LEAF_NAMES, ALL_LEAF_IDS, TIME_GRID = load_tskit_threading_inputs(
#     TREES_PATH,
#     time_grid_size=TIME_GRID_SIZE,
# )

## First example
REFERENCE_FULL_TREES = [
    {
        "sites": (0, 1),
        # "tree": ("n", 2.0, ("n", 1.0, ("n", 1.0, 0, 1), 2), 3),
        "tree": ("n", 2.0, ("n", 0.25, 0, 1), ("n", 0.25, 2, 3)),
    },
    {
        "sites": (1, 8),
        "tree": ("n", 2.0, ("n", 0.25, 0, 2), ("n", 0.25, 1, 3)),
    },
]

GENO = torch.tensor(
    [
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 1, 1, 1],
        [1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1],
    ],
    dtype=torch.long,
)
LEAF_NAMES = ["A", "B", "C", "D"]
ALL_LEAF_IDS = [0, 1, 2, 3]
TIME_GRID = (0.25, 0.5, 1.0, 2.0, 4.0)

In [9]:
ALL_LEAF_IDS[-1]

3

In [ ]:
## Env

N_EPISODES = 5
FOCAL_LEAF_ORDER = [ALL_LEAF_IDS[i % len(ALL_LEAF_IDS)] for i in range(N_EPISODES)]
PRINT_ACTIONS = False
SHOW_LOCAL_TREES = False
env_cfg = ThreadingConfig.from_raw(GENO, TIME_GRID, 0.4, 0.35, 0.15)
env = SingleLeafThreadEnv(env_cfg, ALL_LEAF_IDS, REFERENCE_FULL_TREES)

In [ ]:
## Policy
policy_model = Policy(LEAF_NAMES, device='cpu')

In [ ]:
## N inner episodes with Trajectory Balance and terminal total reward
## One episode = one focal leaf threaded across all sites.


if not hasattr(policy_model, "log_Z"):
    raise RuntimeError("Policy is missing log_Z; rerun the Policy cell after reloading models.py.")
if "tb_loss" not in globals():
    tb_loss = TrajectoryBalanceLoss()


def _optimizer_matches_model(optimizer, model):
    model_params = {id(param) for param in model.parameters()}
    optimizer_params = {id(param) for group in optimizer.param_groups for param in group["params"]}
    return model_params == optimizer_params


if "optimizer" not in globals() or not _optimizer_matches_model(optimizer, policy_model):
    optimizer = torch.optim.Adam(policy_model.parameters(), lr=1e-3)


def _choice_key(choice):
    return choice.branch_child, choice.branch_signature


policy_model.train()
episode_summaries = []

for episode_idx in range(N_EPISODES):
    episode_focal_leaf = FOCAL_LEAF_ORDER[episode_idx]
    st = env.reset(focal_leaf=episode_focal_leaf)
    saved_log_probs = []
    episode_actions = []
    step_log_rewards = []
    done = False
    step_idx = 0

    while not done:
        assert env._inner_env is not None and st.inner_state is not None
        inner_env = env._inner_env
        inner_state = st.inner_state
        site_idx = inner_state.site_index

        valid_acts = env.valid_actions(st)
        if not valid_acts:
            raise RuntimeError(f"No valid actions at episode {episode_idx}, step {step_idx}")

        # Keep action_choices aligned with valid_acts: row j in action_probs maps to valid_acts[j].
        action_choices = [inner_env.site_choices[site_idx][action_idx] for action_idx in valid_acts]
        site_tree = inner_env._site_tree_for_encoding(inner_state)
        action_logits, action_probs, edge_features, node_embeddings, leaf_feature = policy_model(
            site_tree,
            st.current_focal_leaf,
            action_choices,
            time_grid=env_cfg.time_grid,
        )

        focal_name = LEAF_NAMES[st.current_focal_leaf]
        probs_cpu = action_probs.detach().cpu()
        if PRINT_ACTIONS:
            print(f"\n=== episode {episode_idx + 1}/{N_EPISODES} | leaf {focal_name} | site {site_idx} ===")
            print("Action probabilities:")
            for row_idx, (action_idx, prob) in enumerate(zip(valid_acts, probs_cpu)):
                desc = env.describe_action(st, action_idx, leaf_names=LEAF_NAMES)
                print(f"  row {row_idx:02d} -> env action {action_idx:02d} | p={prob.item():.4f} | {desc}")

        dist = torch.distributions.Categorical(probs=action_probs)
        selected_row = dist.sample()
        selected_row_idx = int(selected_row.item())
        selected_action = valid_acts[selected_row_idx]
        selected_choice = action_choices[selected_row_idx]
        selected_desc = env.describe_action(st, selected_action, leaf_names=LEAF_NAMES)
        saved_log_probs.append(dist.log_prob(selected_row))
        if PRINT_ACTIONS:
            print(f"Selected: row {selected_row_idx} -> env action {selected_action} | {selected_desc}")

        # Build a preview state for optional visualization before env.step commits the action.
        recomb_count = inner_state.recomb_count
        if inner_state.choices and _choice_key(selected_choice) != _choice_key(inner_state.choices[-1]):
            recomb_count += 1
        preview_inner_state = type(inner_state)(
            site_index=site_idx + 1,
            choices=inner_state.choices + (selected_choice,),
            recomb_count=recomb_count,
        )

        st, local_log_reward, done = env.step(st, selected_action)
        step_log_rewards.append(float(local_log_reward))

        episode_actions.append(
            {
                "step": step_idx,
                "leaf": focal_name,
                "site": site_idx,
                "row": selected_row_idx,
                "env_action": int(selected_action),
                "prob": float(probs_cpu[selected_row_idx].item()),
                "local_log_reward": float(local_log_reward),
                "description": selected_desc,
            }
        )

        if SHOW_LOCAL_TREES:
            local_trees = inner_env.snapshot_state(preview_inner_state)
            if local_trees:
                draw_local_tree_sequence(
                    local_trees,
                    leaf_names=LEAF_NAMES,
                    time_grid=TIME_GRID,
                    use_time_as_y=True,
                )
                plt.show()

        step_idx += 1

    assert env._inner_env is not None and st.inner_state is not None
    log_probs_t = torch.stack(saved_log_probs)
    sum_log_P_F = log_probs_t.sum()
    step_log_rewards_t = torch.tensor(step_log_rewards, dtype=log_probs_t.dtype, device=log_probs_t.device)
    total_log_reward = torch.tensor(
        env._inner_env.compute_log_reward(st.inner_state),
        dtype=log_probs_t.dtype,
        device=log_probs_t.device,
    )
    additive_log_reward = step_log_rewards_t.sum()
    if not torch.allclose(total_log_reward, additive_log_reward, atol=1e-5):
        raise RuntimeError(
            "Terminal total reward does not match accumulated local rewards: "
            f"{total_log_reward.item():.4f} vs {additive_log_reward.item():.4f}"
        )

    loss = tb_loss(
        policy_model.log_Z,
        sum_log_P_F.reshape(1),
        total_log_reward.reshape(1),
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    summary = {
        "episode": episode_idx + 1,
        "focal_leaf": int(episode_focal_leaf),
        "steps": step_idx,
        "sum_log_P_F": float(sum_log_P_F.item()),
        "total_log_reward": float(total_log_reward.item()),
        "loss": float(loss.item()),
        "leaves_threaded": st.leaves_threaded,
        "actions": episode_actions,
        "full_trees": st.current_full_trees,
    }
    episode_summaries.append(summary)
    print(
        f"episode {episode_idx + 1:02d}/{N_EPISODES} | "
        f"leaf {LEAF_NAMES[episode_focal_leaf]} | steps {step_idx} | "
        f"sum_log_P_F {sum_log_P_F.item():.4f} | "
        f"total_log_reward {total_log_reward.item():.4f} | loss {loss.item():.4f}"
    )

print("\n=== training summary ===")
print(f"episodes: {len(episode_summaries)}")
print("focal leaf order: " + ", ".join(f"{LEAF_NAMES[lid]} ({lid})" for lid in FOCAL_LEAF_ORDER))
print(f"final log_Z: {policy_model.log_Z.item():.4f}")
print(f"final backbone segments: {len(env.current_full_trees)}")
